In [ ]:
"""goal of this code is to take in the full coi sample and per document return only a list of the series, and a ranking for them.
ex: 'Series C > Series A = Series B > Common' should return ['Series C', 'Series A', 'Series B', 'Common'] and [1, 2, 2, 3] or similar. (tbd of exact format)


"""

In [ ]:
import ast
import pandas as pd
import numpy as np


In [ ]:
def parse_expression(expr):
    """
    Parse a single ranking expression string like:
      - 'Series C > Series A = Series B > Common'
      - 'Series A > Series 1 > Common'
      - 'C-1 > C > B > A > Common'
      - 'Series F > Series E=D=C > ...'

    Returns a list of lists (tiers), e.g.:
      [['Series C'], ['Series A', 'Series B'], ['Common']]
    """
    expr = str(expr).strip()
    if not expr:
        return []

    # Split on ">" into ordered groups
    raw_parts = [p.strip() for p in expr.split('>') if p.strip()]

    tiers = []
    for part in raw_parts:
        # Within each part, split on "=" to handle same-rank securities
        subparts = [x.strip() for x in part.split('=') if x.strip()]
        tiers.append(subparts)

    return tiers


In [ ]:
def parse_any_cell(val):
    """
    Parse a single cell from your target column into tier structure.

    Returns:
      - list of lists (tiers) if parseable
      - None for MISSING / Uncertain / NaN
    """
    # Handle explicit missing markers
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return None

    if isinstance(val, str) and val.strip().upper() in {"MISSING", "UNCERTAIN"}:
        return None

    # Try to interpret string as a Python literal (e.g. "['Series A > Common']")
    v = val
    if isinstance(val, str):
        try:
            v = ast.literal_eval(val)
        except (SyntaxError, ValueError):
            v = val  # not a literal, treat as raw string

    tiers = []

    if isinstance(v, list):
        # Each element of the list is its own entry: could be expression or single label
        for item in v:
            if isinstance(item, str) and '>' in item:
                # Full ordering expression within the list
                tiers.extend(parse_expression(item))
            elif isinstance(item, str) and '=' in item:
                # Same-rank block like 'D=D-1' or 'A=B=C=D'
                group = [x.strip() for x in item.split('=') if x.strip()]
                tiers.append(group)
            else:
                # Single label, already in order given by the list
                label = item.strip() if isinstance(item, str) else item
                if label:
                    tiers.append([label])

    elif isinstance(v, str):
        # Raw string expression like "Series C > Series B > Series A3 > Common Stock"
        if '>' in v or '=' in v:
            tiers = parse_expression(v)
        else:
            # Single label in a string, e.g. "Common"
            label = v.strip()
            if label:
                tiers = [[label]]
            else:
                return None
    else:
        # Unknown format
        return None

    return tiers if tiers else None


In [ ]:
def normalize_label(label):
    """
    Optional: normalize labels for consistent matching across datasets.
    - Trim whitespace
    - Collapse 'Common Stock' to 'Common'
    You can add more rules here if needed.
    """
    lab = str(label).strip()
    if lab.lower() == "common stock":
        return "Common"
    return lab


def build_rank_map_from_tiers(tiers, normalize=True):
    """
    Given tiers as list of lists, build dict: label -> rank (1 = highest).
    """
    if not tiers:
        return {}

    rank_map = {}
    for rank, group in enumerate(tiers, start=1):
        for label in group:
            key = normalize_label(label) if normalize else label
            rank_map[key] = rank

    return rank_map
